In [41]:
from astroquery.alma import Alma
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
import numpy as np
import re
from astropy.table import Table

In [2]:
ASA_query = Alma()
ASA_query.archive_url = "https://almascience.nao.ac.jp"

In [3]:
def parse_frequency_support(frequency_support_str):
    spw_list = str(frequency_support_str).split("U")
    freq_range_list = []
    for spw in spw_list:
        numin, numax = spw.strip()[1:-1].split(",")[0].split("..")
        numax = numax.replace("GHz", "")
        freq_range_list.append((float(numin), float(numax)))
    return freq_range_list

    

def format_ALMA_query(query_result):
    df = pd.DataFrame()
    df["project_code"] = query_result["proposal_id"]
    df["source_name"] = query_result["target_name"]

    # source coordinate
    coord = SkyCoord(ra=query_result["s_ra"], dec=query_result["s_dec"], unit=u.deg)
    df["R.A."] = coord.ra.to_string(sep="hms", precision=3)
    df["Dec."] = coord.dec.to_string(sep="dms", precision=2)
    df["Band"] = query_result["band_list"]

    # frequency coverage
    freq_support = []
    for i, row in query_result.iterrows():
        freq_range_list = parse_frequency_support(row["frequency_support"])
        numin, numax = np.min(freq_range_list), np.max(freq_range_list)
        freq_support.append(f"{numin:.2f}–{numax:.2f} GHz")
    df["Freq. Support"] = freq_support

    df["Ang. Res. (arcsec)"] = query_result["s_resolution"]
    df["Min. Vel. Res. (km/s)"] = query_result["velocity_resolution"] * 1e-3
    df["Line Sens. @ 10 km/s (mJy/beam)"] = query_result["sensitivity_10kms"]
    df["Int. Time (h)"] = query_result["t_exptime"] / 3600 # in hour
    df["PWV (mm)"] = query_result["pwv"]
    df["PI"] = query_result["pi_name"]
    df["Status"] = query_result["data_rights"]

    # resolve duplication and averaged sensitivity
    # subset = df[df.duplicated(subset=["project_code", "source_name", "Freq. Support", "Ang. Res. (arcsec)"], keep=False)]
    # print(subset)
    df = df.loc[
        df.groupby(["project_code", "source_name", "Freq. Support", "Ang. Res. (arcsec)"])["Line Sens. @ 10 km/s (mJy/beam)"].idxmin()
    ]

    df = df.sort_values("project_code")
    
    # return df.drop_duplicates()
    return df


In [9]:
def read_CDMS_partition_function(filename, tag):
    """
    Parameters
    ----------
    filename : str
        ファイル名
    mol_id : int
        取得したいID（最左列）

    Returns
    -------
    temps : np.ndarray
        有効な温度のみの配列
    Qvals : np.ndarray
        対応する log10(Q) の配列（--- は除外）
    """

    temps = None

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()

            # ヘッダーから温度取得
            if line.startswith("tag"):
                temps_all = np.array(
                    [float(t) for t in re.findall(r"Q\(([\d\.]+)\)", line)]
                )

            # データ行
            elif line and not line.startswith("="):
                parts = line.split()

                try:
                    current_id = int(parts[0])
                except ValueError:
                    continue

                if current_id == tag:
                    Qvals = []
                    temps = []

                    for t, v in zip(temps_all, parts[3:]):
                        if v != "---":
                            temps.append(t)
                            Qvals.append(float(v))

                    return np.array(temps), 10 ** np.array(Qvals)

    raise ValueError(f"ID {tag} not found")

def read_JPL_partition_function(filename, tag):
    """
    Parameters
    ----------
    filename : str
    mol_id : int

    Returns
    -------
    temps : np.ndarray
        温度配列（昇順）
    Qvals : np.ndarray
        log10(Q)
    """

    # 温度（昇順）
    temps = np.array([9.375, 18.75, 37.50, 75.00, 150.0, 225.0, 300.0])

    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()

            try:
                current_id = int(parts[0])
            except ValueError:
                continue

            if current_id == tag:
                # 4列目から「最後の1列手前まで」を使う
                Qvals = np.array([float(v) for v in parts[3:-1]])

                # 元は高温→低温なので反転
                Qvals = Qvals[::-1]

                return temps, 10 ** Qvals

    raise ValueError(f"ID {tag} not found")

In [50]:
import astropy.io.ascii as ascii

filename = "./database/partition_function.dat"
def read_CDMS_partition_function(filename, tag):
    with open(filename, "r") as f:
        data = f.read()

    lines = data.split("\n")
    def tryfloat(x):
            try:
                return float(x)
            except ValueError:
                return np.nan

    # the 'fixed width' table reader fails because there are rows that violate fixed width
    tbl_rows = []
    for row in lines[15:-5]:
        split = row.split()
        tag = int(split[0])
        molecule_and_lines = row[7:41]
        molecule = " ".join(molecule_and_lines.split()[:-1])
        nlines = int(molecule_and_lines.split()[-1])
        partfunc = map(tryfloat, row[41:].split())
        partfunc_dict = dict(zip(['lg(Q(1000))', 'lg(Q(500))', 'lg(Q(300))', 'lg(Q(225))',
                                    'lg(Q(150))', 'lg(Q(75))', 'lg(Q(37.5))', 'lg(Q(18.75))',
                                    'lg(Q(9.375))', 'lg(Q(5.000))', 'lg(Q(2.725))'], partfunc))
        tbl_rows.append({'tag': tag,
                            'molecule': molecule,
                            '#lines': nlines,
                            })
        tbl_rows[-1].update(partfunc_dict)
    tbl = Table(tbl_rows)

    temps = np.array([100, 500, 300, 225, 150, 75, 37.5, 18.75, 9.375, 5.000, 2.725])
    Qvals = tbl[tbl["tag"] == tag]
    Qvals = np.array(list(Qvals[0])[3:])
    # print(tbl)
    return temps[~np.isnan(Qvals)], 10 ** Qvals[~np.isnan(Qvals)]

t, q = read_CDMS_partition_function(filename, tag=18501)
t, q

(array([300.   , 225.   , 150.   ,  75.   ,  37.5  ,  18.75 ,   9.375,
          5.   ,   2.725]),
 array([8.36372992e+05, 5.43125258e+05, 2.95665057e+05, 1.04520144e+05,
        3.69572800e+04, 1.30707347e+04, 4.62381021e+03, 1.80260263e+03,
        7.26607707e+02]))

In [53]:
def get_CDMS_table(filename):
    with open(filename, "r") as f:
        data = f.read()

    lines = data.split("\n")
    def tryfloat(x):
            try:
                return float(x)
            except ValueError:
                return np.nan

    # the 'fixed width' table reader fails because there are rows that violate fixed width
    tbl_rows = []
    for row in lines[15:-5]:
        split = row.split()
        tag = int(split[0])
        molecule_and_lines = row[7:41]
        molecule = " ".join(molecule_and_lines.split()[:-1])
        nlines = int(molecule_and_lines.split()[-1])
        partfunc = map(tryfloat, row[41:].split())
        partfunc_dict = dict(zip(['lg(Q(1000))', 'lg(Q(500))', 'lg(Q(300))', 'lg(Q(225))',
                                    'lg(Q(150))', 'lg(Q(75))', 'lg(Q(37.5))', 'lg(Q(18.75))',
                                    'lg(Q(9.375))', 'lg(Q(5.000))', 'lg(Q(2.725))'], partfunc))
        tbl_rows.append({'tag': tag,
                            'name': molecule,
                            '#lines': nlines,
                            })
        tbl_rows[-1].update(partfunc_dict)
    tbl = Table(tbl_rows)
    return tbl


def fetch_CDMS_species():
    tbl = get_CDMS_table(filename)
    df_mol = tbl["tag", "name"].to_pandas()
    # df_mol = pd.read_csv(
    #     CDMS_PF_filename,
    #     sep='\s+', 
    #     skip_blank_lines=True,
    #     skiprows=4,
    #     usecols=[0,1],
    #     names=["tag", "name"]
    # )
    df_mol["catalog"] = "CDMS"
    return df_mol

In [54]:
fetch_CDMS_species()

,tag,name,catalog
0,13505,"CH+, v=2-0",CDMS
1,13506,C-13-+,CDMS
2,14501,CH2,CDMS
3,14502,C-13-H+,CDMS
4,14503,CD+,CDMS
5,14504,"C-13-H+, v=1-0",CDMS
6,14505,"CD+, v=1-0",CDMS
7,14506,N+,CDMS
8,15501,NH,CDMS
9,15502,C-13-D+,CDMS


In [14]:
# get partition function
from specdata import SpectroscopicData, PartitionFunction
name = "NH2D"
tag = 18004
catalog = "JPL"

JPL_PF_filename = "./database/catdir.cat"
CDMS_PF_filename = "./database/partition_function.dat"

PF_FILENAME = JPL_PF_filename if catalog == "JPL" else CDMS_PF_filename
# T, Q = getattr(SpectroscopicData, f"read_{catalog}_partition_function")(PF_FILENAME, tag)
T, Q = read_JPL_partition_function(PF_FILENAME, tag)
pf = PartitionFunction(species=name, T=T, Q=Q, database=catalog)

filename = f"./database/{catalog}/c{str(tag).zfill(6)}.cat"
specdata = SpectroscopicData(filename=filename, format=catalog, species=name, pf=pf)
# specdata.table

# filtering with Eup and logint
Eumin, Eumax = (0.0, 500)
if Eumin is None: Eumin = 0.0
if Eumax is None: Eumax = np.inf

logintmin, logintmax = (-8, 0)
if logintmin is None: logintmin = -np.inf
if logintmax is None: logintmax = np.inf
specdata.table = specdata.table[(specdata.table["E_up"] >= Eumin) & (specdata.table["E_up"] <= Eumax) & (specdata.logint >= logintmin) & (specdata.logint <= logintmax)]

In [ ]:
source_name = "V883 Ori"
data_all = ASA_query.query_region(source_name, radius=1.0 * u.arcmin)#.to_pandas()
data_all
# df = format_ALMA_query(data_all)
# data_all["frequency_support"][0]
# df